# PennyLane QAOA for MaxCut

Evaluate a p=1 triangle-graph QAOA landscape with the same Hamiltonian on both devices.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
cost = 1.5 * qml.I(0) - 0.5 * (qml.Z(0) @ qml.Z(1)) - 0.5 * (qml.Z(1) @ qml.Z(2)) - 0.5 * (qml.Z(0) @ qml.Z(2))
parameters = [(gamma, beta) for gamma in np.linspace(0.0, np.pi, 7) for beta in np.linspace(0.0, np.pi / 2, 5)]

def make_qnode(device):
    @qml.qnode(device)
    def circuit(gamma, beta):
        for wire in range(3):
            qml.Hadamard(wire)
        for wires in ((0, 1), (1, 2), (0, 2)):
            qml.IsingZZ(-gamma, wires=wires)
        for wire in range(3):
            qml.RX(2 * beta, wires=wire)
        return qml.expval(cost)
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=3))
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(*values) for values in parameters]))
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(*values) for values in parameters]))
error = max_abs_error(reference, candidate)
best_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/06_qaoa_maxcut.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="QAOA landscape atol=4e-6",
    passed=error <= 4e-6 and best_match,
    exact_match=best_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_cost_error": error, "best_parameters": parameters[int(np.argmax(candidate))], "best_cost": candidate.max()},
)